# Module 6 / Class 3 -- CNN on MNIST

**Objectives:**
- Load and explore the MNIST handwritten digits dataset
- Build a Dense-only baseline model
- Build a CNN and compare performance
- Visualize training curves

**Runtime:** Use GPU if available (Runtime > Change runtime type > GPU).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Load and Explore MNIST

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Training set:  {X_train.shape}, labels: {y_train.shape}")
print(f"Test set:      {X_test.shape}, labels: {y_test.shape}")
print(f"Pixel range:   [{X_train.min()}, {X_train.max()}]")
print(f"Label classes:  {np.unique(y_train)}")

In [ ]:
# Show sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.suptitle('Sample MNIST Images')
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(8, 4))
plt.bar(unique, counts, edgecolor='black')
plt.title('Class Distribution (Training Set)')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.xticks(unique)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

## 2. Preprocessing

In [ ]:
# Normalize pixel values to [0, 1]
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

# Reshape for CNN: (samples, height, width, channels)
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
X_test_cnn = X_test_norm.reshape(-1, 28, 28, 1)

print(f"Flat shape for Dense model:  {X_train_norm.shape}")
print(f"CNN shape:                   {X_train_cnn.shape}")

## 3. Baseline -- Dense-Only Model

In [ ]:
dense_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

dense_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

dense_model.summary()

In [ ]:
dense_history = dense_model.fit(
    X_train_norm, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)

In [ ]:
dense_test_loss, dense_test_acc = dense_model.evaluate(X_test_norm, y_test, verbose=0)
print(f"Dense model -- Test accuracy: {dense_test_acc:.4f}")

## 4. CNN Model

In [ ]:
cnn_model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

cnn_model.summary()

In [ ]:
cnn_history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"CNN model -- Test accuracy: {cnn_test_acc:.4f}")

## 5. Training Curves

In [ ]:
def plot_training_curves(history, title):
    """Plot accuracy and loss curves."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0].set_title(f'{title} -- Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'], label='Train', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[1].set_title(f'{title} -- Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_training_curves(dense_history, 'Dense Model')
plot_training_curves(cnn_history, 'CNN Model')

## 6. Comparison

In [ ]:
print("=" * 40)
print(f"{'Model':<15} {'Test Accuracy':>15}")
print("=" * 40)
print(f"{'Dense':<15} {dense_test_acc:>15.4f}")
print(f"{'CNN':<15} {cnn_test_acc:>15.4f}")
print("=" * 40)

## 7. Visualize Some Predictions

In [ ]:
predictions = cnn_model.predict(X_test_cnn[:20], verbose=0)
pred_labels = np.argmax(predictions, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i], cmap='gray')
    color = 'green' if pred_labels[i] == y_test[i] else 'red'
    ax.set_title(f'Pred: {pred_labels[i]} (True: {y_test[i]})', color=color)
    ax.axis('off')
plt.suptitle('CNN Predictions (green=correct, red=wrong)')
plt.tight_layout()
plt.show()

---

## TODO: Student Assignment -- Add Dropout and Compare

Build a third model that is identical to the CNN above, but add `Dropout(0.25)` after each MaxPooling layer and `Dropout(0.5)` before the final Dense layer.

1. Build and train the model (5 epochs)
2. Plot training curves
3. Compare test accuracy with the original CNN
4. Answer: Did Dropout help? Why or why not on this dataset?

In [ ]:
# TODO: Build CNN with Dropout
# cnn_dropout_model = keras.Sequential([
#     layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
#     layers.MaxPooling2D((2, 2)),
#     layers.Dropout(0.25),
#     ... add remaining layers ...
# ])
#
# cnn_dropout_model.compile(...)
# dropout_history = cnn_dropout_model.fit(...)

In [ ]:
# TODO: Evaluate and compare
# dropout_test_loss, dropout_test_acc = cnn_dropout_model.evaluate(X_test_cnn, y_test, verbose=0)
# print(f"CNN + Dropout -- Test accuracy: {dropout_test_acc:.4f}")

# TODO: Write your analysis

- CNN test accuracy: ...
- CNN + Dropout test accuracy: ...
- Did Dropout help? Why or why not?